# GRAFT Phase 4: Evaluation and Metrics
Compute Comprehensiveness, Diversity, Faithfulness, ROUGE, and BERTScore on the holdout test set to compare vector RAG, GraphRAG, and GRAFT.

In [ ]:
import sys
import os
import json

sys.path.append(os.path.join("..", "src"))
from evaluator import Evaluator
from graft_inference import GRAFTAnswer

eval_suite = Evaluator()

# 1. Mock Data for demonstration
answers_a = ["GRAFT provides a detailed explanation incorporating global sensemaking."]
answers_b = ["Baseline RAFT provides a localized lookup without community context."]
questions = ["What are the central themes of the text?"]

print("Loaded Mock Data")

In [ ]:
# 2. Setup Evaluation Metric Calculations
# LLM pairwise judge simulating win-rates
win_rates = eval_suite.llm_pairwise_judge(answers_a, answers_b, questions)
print("Pairwise Win-Rates:", win_rates)

In [ ]:
# 3. Assess Fact/Claim Extraction Quality (Comprehensiveness)
graft_ans = GRAFTAnswer(
    relevant_docs=["C1"], 
    reasoning_chain="Chain 1", 
    final_answer=answers_a[0] + " Additional fact here.",
    citations=["[C1]"]
)

claim_metrics = eval_suite.claim_metrics([graft_ans], ["Gold Standard Answer"], distance_thresholds=[0.5, 0.7])
print("Claim Extraction Metrics:\n", json.dumps(claim_metrics, indent=2))

In [ ]:
# 4. Standard NLP Similarity Metrics
test_gen = ["The system evaluates successfully with high variance."]
test_ref = ["A system successfully evaluates with varied output."]
nlp_res = eval_suite.standard_nlp_metrics(test_gen, test_ref)

print("Standard NLP Result:\n", json.dumps(nlp_res, indent=2))

In [ ]:
# 5. Build full result payload and store
eval_suite.save_results("GRAFT", {
    "Comprehensiveness": claim_metrics["comprehensiveness_avg_claims"],
    "Citation Precision": claim_metrics["citation_precision"],
    **nlp_res
})
print("Metrics Extracted & Stored!")